In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType, LongType
)

# Sesión Spark
spark = (
    SparkSession.builder
    .appName("wrangler_temperatura_01")
    .config("spark.sql.session.timeZone", "America/Lima")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

print("Spark listo:", spark.version)

Spark listo: 3.5.3


Celda 2 — Rutas de trabajo

In [2]:
# Rutas base (dentro del contenedor, /home/jovyan/work apunta a D:\aireiq-bigdata)
RAW       = "/home/jovyan/work/data/raw/mediciones_aire.csv"
BRONZE    = "/home/jovyan/work/data/bronze/wrangler_temperatura"
SILVER    = "/home/jovyan/work/data/silver/wrangler_temperatura"
GOLD      = "/home/jovyan/work/data/gold/wrangler_temperatura"

# Verificar el CSV 
import os
existe = os.path.exists(RAW)
print("CSV encontrado:", existe)
print("Ruta:", RAW)

CSV encontrado: True
Ruta: /home/jovyan/work/data/raw/mediciones_aire.csv


In [3]:
# Esquema explícito (igual que Frank, adaptado a temperatura)
schema = StructType([
    StructField("id",                   LongType(),      True),
    StructField("created_at",           StringType(),    True),
    StructField("dispositivo",          StringType(),    True),
    StructField("temperatura_c",        DoubleType(),    True),
    StructField("humedad_pct",          DoubleType(),    True),
    StructField("presion_hpa",          DoubleType(),    True),
    StructField("gas_ohms",             DoubleType(),    True),
    StructField("altura_m",             DoubleType(),    True),
    StructField("iaq_estimado",         DoubleType(),    True),
    StructField("eco2_estimado_ppm",    DoubleType(),    True),
    StructField("voc_indice_estimado",  DoubleType(),    True),
    StructField("calidad_aire",         StringType(),    True),
])

# Leer CSV tal cual (capa bronce)
df_bronze = (
    spark.read
    .option("header", "true")
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss.SSSSSSx")
    .schema(schema)
    .csv(RAW)
)

total_bronze = df_bronze.count()
print(f"Filas en bronce: {total_bronze:,}")
df_bronze.printSchema()

Filas en bronce: 18,092
root
 |-- id: long (nullable = true)
 |-- created_at: string (nullable = true)
 |-- dispositivo: string (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- humedad_pct: double (nullable = true)
 |-- presion_hpa: double (nullable = true)
 |-- gas_ohms: double (nullable = true)
 |-- altura_m: double (nullable = true)
 |-- iaq_estimado: double (nullable = true)
 |-- eco2_estimado_ppm: double (nullable = true)
 |-- voc_indice_estimado: double (nullable = true)
 |-- calidad_aire: string (nullable = true)



In [4]:
# Guardar bronce particionado por fecha
(
    df_bronze
    .write
    .mode("overwrite")
    .partitionBy("dispositivo")
    .parquet(BRONZE)
)

# Verificar ida y vuelta
df_bronze_check = spark.read.parquet(BRONZE)
total_check = df_bronze_check.count()
print(f"Bronce guardado: {total_check:,} filas")
print("✓ Ida y vuelta OK" if total_check == total_bronze else "✗ Discrepancia")

Bronce guardado: 18,092 filas
✓ Ida y vuelta OK


In [6]:
# Parsear timestamp con zona Lima directamente
df_parsed = (
    df_bronze
    .withColumn(
        "ts",
        F.to_timestamp(F.col("created_at"), "yyyy-MM-dd HH:mm:ss.SSSSSSx")
    )
)

# El timestamp ya tiene offset -05 en el string, to_timestamp lo respeta.
# Forzamos la sesión a Lima para que las funciones de hora sean correctas.
spark.conf.set("spark.sql.session.timeZone", "America/Lima")

# 1. Descartar filas con gas_ohms = 0 (sensor fallido)
n_antes = df_parsed.count()
df_sin_fallo = df_parsed.filter(F.col("gas_ohms") > 0)
n_fallos = n_antes - df_sin_fallo.count()
print(f"Filas con gas_ohms=0 descartadas: {n_fallos:,}")

# 2. Descartar duplicados por timestamp
df_sin_dup = df_sin_fallo.dropDuplicates(["ts"])
n_dup = df_sin_fallo.count() - df_sin_dup.count()
print(f"Duplicados descartados: {n_dup:,}")

# 3. Filtrar rangos físicos válidos de temperatura
df_silver_raw = df_sin_dup.filter(
    (F.col("temperatura_c") >= 13.0) &
    (F.col("temperatura_c") <= 35.0)
)
n_rango = df_sin_dup.count() - df_silver_raw.count()
print(f"Fuera de rango físico descartadas: {n_rango:,}")
print(f"Filas válidas antes de remuestrear: {df_silver_raw.count():,}")

Filas con gas_ohms=0 descartadas: 2,851
Duplicados descartados: 0
Fuera de rango físico descartadas: 0
Filas válidas antes de remuestrear: 15,241


In [7]:
# Truncar al minuto y promediar
df_silver = (
    df_silver_raw
    .withColumn("minuto", F.date_trunc("minute", F.col("ts")))
    .groupBy("minuto")
    .agg(
        F.avg("temperatura_c").alias("temperatura_c"),
        F.avg("humedad_pct").alias("humedad_pct"),
        F.avg("presion_hpa").alias("presion_hpa"),
        F.avg("gas_ohms").alias("gas_ohms"),
        F.count("*").alias("n_lecturas")
    )
    .orderBy("minuto")
)

total_silver = df_silver.count()
print(f"Minutos únicos en plata: {total_silver:,}")
df_silver.show(5, truncate=False)

Minutos únicos en plata: 4,862
+-------------------+-------------+-----------+-----------+--------+----------+
|minuto             |temperatura_c|humedad_pct|presion_hpa|gas_ohms|n_lecturas|
+-------------------+-------------+-----------+-----------+--------+----------+
|2026-10-21 23:52:00|18.7746      |37.3878    |647.35     |398474.0|1         |
|2026-10-21 23:53:00|18.7791      |37.3368    |647.34     |421737.0|1         |
|2026-10-21 23:54:00|18.8374      |37.251     |647.33     |425476.0|1         |
|2026-10-21 23:55:00|18.8675      |37.2025    |647.33     |441992.0|1         |
|2026-10-21 23:56:00|18.9174      |37.1502    |647.32     |438244.0|1         |
+-------------------+-------------+-----------+-----------+--------+----------+
only showing top 5 rows



In [8]:
# Guardar plata
(
    df_silver
    .write
    .mode("overwrite")
    .parquet(SILVER)
)

# Verificar ida y vuelta
df_silver_check = spark.read.parquet(SILVER)
total_check = df_silver_check.count()
print(f"Plata guardada: {total_check:,} filas")
print("✓ Ida y vuelta OK" if total_check == total_silver else "✗ Discrepancia")

Plata guardada: 4,862 filas
✓ Ida y vuelta OK


In [9]:
# Periodo de referencia: 29 oct – 1 nov 2026
df_ref = df_silver.filter(
    (F.col("minuto") >= "2026-10-29") &
    (F.col("minuto") <  "2026-11-02")
)

n_ref = df_ref.count()
print(f"Minutos en periodo de referencia: {n_ref:,}")

# Calcular p5 y p95
percentiles = df_ref.select(
    F.percentile_approx("temperatura_c", 0.05).alias("p5"),
    F.percentile_approx("temperatura_c", 0.95).alias("p95"),
    F.min("temperatura_c").alias("temp_min"),
    F.max("temperatura_c").alias("temp_max"),
    F.avg("temperatura_c").alias("temp_media"),
).collect()[0]

P5  = percentiles["p5"]
P95 = percentiles["p95"]

print(f"\nTemperatura en el periodo de referencia:")
print(f"  Mínima:  {percentiles['temp_min']:.2f} °C")
print(f"  p5:      {P5:.2f} °C  ← límite inferior")
print(f"  Media:   {percentiles['temp_media']:.2f} °C")
print(f"  p95:     {P95:.2f} °C  ← límite superior")
print(f"  Máxima:  {percentiles['temp_max']:.2f} °C")

Minutos en periodo de referencia: 1,277

Temperatura en el periodo de referencia:
  Mínima:  13.60 °C
  p5:      16.53 °C  ← límite inferior
  Media:   20.97 °C
  p95:     26.62 °C  ← límite superior
  Máxima:  27.89 °C


In [10]:
# Agregar columnas de hora y fecha
df_con_tiempo = df_silver.withColumn(
    "hora", F.hour("minuto")
).withColumn(
    "fecha", F.to_date("minuto")
).withColumn(
    "fuera_rango", 
    F.when(
        (F.col("temperatura_c") < P5) | (F.col("temperatura_c") > P95), 1
    ).otherwise(0)
).withColumn(
    "sobre_p95", F.when(F.col("temperatura_c") > P95, 1).otherwise(0)
).withColumn(
    "bajo_p5",   F.when(F.col("temperatura_c") < P5,  1).otherwise(0)
)

# --- Tabla oro 1: indicadores por hora del día ---
df_oro_hora = (
    df_con_tiempo
    .groupBy("hora")
    .agg(
        F.count("*").alias("n_minutos"),
        F.avg("temperatura_c").alias("temp_media"),
        F.min("temperatura_c").alias("temp_min"),
        F.max("temperatura_c").alias("temp_max"),
        (F.sum("fuera_rango") / F.count("*") * 100).alias("pct_fuera_rango"),
        (F.sum("sobre_p95")   / F.count("*") * 100).alias("pct_sobre_p95"),
        (F.sum("bajo_p5")     / F.count("*") * 100).alias("pct_bajo_p5"),
    )
    .orderBy("hora")
)

# --- Tabla oro 2: indicadores por día ---
df_oro_dia = (
    df_con_tiempo
    .groupBy("fecha")
    .agg(
        F.count("*").alias("n_minutos"),
        F.avg("temperatura_c").alias("temp_media"),
        F.min("temperatura_c").alias("temp_min"),
        F.max("temperatura_c").alias("temp_max"),
        (F.sum("fuera_rango") / F.count("*") * 100).alias("pct_fuera_rango"),
        (F.sum("sobre_p95")   / F.count("*") * 100).alias("pct_sobre_p95"),
        (F.sum("bajo_p5")     / F.count("*") * 100).alias("pct_bajo_p5"),
    )
    .orderBy("fecha")
)

print("Indicadores por hora (primeras 6 horas):")
df_oro_hora.show(6, truncate=False)

print("Indicadores por día:")
df_oro_dia.show(30, truncate=False)

Indicadores por hora (primeras 6 horas):
+----+---------+------------------+--------+------------------+------------------+-------------+------------------+
|hora|n_minutos|temp_media        |temp_min|temp_max          |pct_fuera_rango   |pct_sobre_p95|pct_bajo_p5       |
+----+---------+------------------+--------+------------------+------------------+-------------+------------------+
|0   |303      |20.57124840484049 |16.8961 |23.559433333333335|0.0               |0.0          |0.0               |
|1   |252      |20.238574636243378|16.8097 |22.476575         |0.0               |0.0          |0.0               |
|2   |239      |19.900818758716873|16.7056 |22.106175         |0.0               |0.0          |0.0               |
|3   |255      |19.301395261437914|16.412  |21.709525         |8.235294117647058 |0.0          |8.235294117647058 |
|4   |204      |19.05284350490196 |16.1402 |21.213224999999998|27.450980392156865|0.0          |27.450980392156865|
|5   |221      |18.723796907993

Celda 10 — Guardar capa oro en Parquet

In [11]:
# Guardar tablas de oro
(
    df_oro_hora
    .write.mode("overwrite")
    .parquet(GOLD + "/por_hora")
)

(
    df_oro_dia
    .write.mode("overwrite")
    .parquet(GOLD + "/por_dia")
)

# Verificar ida y vuelta
n_hora = spark.read.parquet(GOLD + "/por_hora").count()
n_dia  = spark.read.parquet(GOLD + "/por_dia").count()

print(f"Oro por_hora: {n_hora} horas guardadas")
print(f"Oro por_dia:  {n_dia} días guardados")

# Guardar también los umbrales como referencia
print(f"\nUmbrales de referencia guardados en memoria:")
print(f"  P5  = {P5:.4f} °C")
print(f"  P95 = {P95:.4f} °C")

Oro por_hora: 24 horas guardadas
Oro por_dia:  14 días guardados

Umbrales de referencia guardados en memoria:
  P5  = 16.5336 °C
  P95 = 26.6225 °C


Celda 11 — Guardar umbrales en JSON

In [12]:
import json

umbrales = {
    "p5":  round(P5,  4),
    "p95": round(P95, 4),
    "periodo_referencia_inicio": "2026-10-29",
    "periodo_referencia_fin":    "2026-11-01",
    "variable": "temperatura_c",
    "unidad": "°C"
}

ruta_umbrales = "/home/jovyan/work/models/wrangler_temperatura/umbrales.json"

with open(ruta_umbrales, "w") as f:
    json.dump(umbrales, f, indent=2, ensure_ascii=False)

# Verificar
with open(ruta_umbrales) as f:
    print(f.read())

{
  "p5": 16.5336,
  "p95": 26.6225,
  "periodo_referencia_inicio": "2026-10-29",
  "periodo_referencia_fin": "2026-11-01",
  "variable": "temperatura_c",
  "unidad": "°C"
}


Celda 12 — Resumen final del notebook 01

In [13]:
print("=" * 55)
print("RESUMEN NOTEBOOK 01 — WRANGLER TEMPERATURA")
print("=" * 55)
print(f"\nCapas generadas:")
print(f"  Bronce : {total_bronze:>6,} filas  (CSV crudo)")
print(f"  Plata  : {total_silver:>6,} filas  (minutos limpios)")
print(f"  Oro    : {n_hora:>6} horas  + {n_dia} días")
print(f"\nFiltros aplicados en plata:")
print(f"  gas_ohms=0 (sensor fallido) : {n_fallos:,} filas descartadas")
print(f"  Duplicados                  : {n_dup} filas descartadas")
print(f"  Fuera de rango físico       : {n_rango} filas descartadas")
print(f"\nUmbrales de referencia (29 oct – 1 nov 2026):")
print(f"  p5  = {P5:.4f} °C  (inusualmente frío por debajo)")
print(f"  p95 = {P95:.4f} °C  (inusualmente caliente por encima)")
print(f"\nArchivos generados:")
print(f"  data/bronze/wrangler_temperatura/")
print(f"  data/silver/wrangler_temperatura/")
print(f"  data/gold/wrangler_temperatura/por_hora/")
print(f"  data/gold/wrangler_temperatura/por_dia/")
print(f"  models/wrangler_temperatura/umbrales.json")
print("\n✓ Notebook 01 completo")

RESUMEN NOTEBOOK 01 — WRANGLER TEMPERATURA

Capas generadas:
  Bronce : 18,092 filas  (CSV crudo)
  Plata  :  4,862 filas  (minutos limpios)
  Oro    :     24 horas  + 14 días

Filtros aplicados en plata:
  gas_ohms=0 (sensor fallido) : 2,851 filas descartadas
  Duplicados                  : 0 filas descartadas
  Fuera de rango físico       : 0 filas descartadas

Umbrales de referencia (29 oct – 1 nov 2026):
  p5  = 16.5336 °C  (inusualmente frío por debajo)
  p95 = 26.6225 °C  (inusualmente caliente por encima)

Archivos generados:
  data/bronze/wrangler_temperatura/
  data/silver/wrangler_temperatura/
  data/gold/wrangler_temperatura/por_hora/
  data/gold/wrangler_temperatura/por_dia/
  models/wrangler_temperatura/umbrales.json

✓ Notebook 01 completo
